<a href="https://colab.research.google.com/github/Karsuman4298/Computer_Vision/blob/main/DeiT.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
#@title Imports
import torch,torch.nn as nn,torch.nn.functional as F
import torchvision
from torchvision import transforms,models,datasets
from torch.utils.data import DataLoader
import numpy as np
import matplotlib.pyplot as plt


In [2]:
#@title device
device ="cuda" if torch.cuda.is_available() else "cpu"
device

'cuda'

In [3]:
#@title Variable
BATCH_SIZE=12
IMG_SIZE=28
PATCH_SIZE=7
ATTENTION_HEADS=4
EMBED_DIM=32
NUM_CLASSES=10
EPOCHS_STUDENT=5
LR_STUDENT=3e-4
CHANNELS=3
TRANSFOMER_LAYERS=4
TEMPERATURE=2
ALPHA=0.5


In [4]:
#@title Import the Data
data_transformations=transforms.Compose([transforms.ToTensor(),
                                         transforms.Lambda(lambda t:t.repeat(3,1,1))
                                         ])
train_ds=datasets.MNIST("./data",train=True,download=True,transform=data_transformations)
val_ds=datasets.MNIST("./data",train=False,download=True,transform=data_transformations)

print(train_ds)

100%|██████████| 9.91M/9.91M [00:00<00:00, 18.3MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 486kB/s]
100%|██████████| 1.65M/1.65M [00:00<00:00, 4.66MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 11.3MB/s]

Dataset MNIST
    Number of datapoints: 60000
    Root location: ./data
    Split: Train
    StandardTransform
Transform: Compose(
               ToTensor()
               Lambda()
           )


In [5]:
#@title Create train and val Batches


train_dl=DataLoader(train_ds,batch_size=BATCH_SIZE,shuffle=True)
val_dl=DataLoader(val_ds,batch_size=BATCH_SIZE)


In [6]:
#@title Teacher model

teacher=models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V2)
teacher.fc=nn.Linear(teacher.fc.in_features,NUM_CLASSES)#we remove the last layer of resnet50 because it has 1000 class classification but we have 10 class classification(in_feature will be same as in teacher model:resnet50)
teacher.to(device)

Downloading: "https://download.pytorch.org/models/resnet50-11ad3fa6.pth" to /root/.cache/torch/hub/checkpoints/resnet50-11ad3fa6.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 121MB/s]


ResNet(
  (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): Bottleneck(
      (conv1): Conv2d(64, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (conv3): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 1), bias=False)
      (bn3): BatchNorm2d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (downsample): Sequential(
        (0): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 

In [ ]:
#@title Student Model

#Patch Embedding
class patch_embed(nn.Module):
  def __init__(self,classes=NUM_CLASSES,embed_dim=EMBED_DIM,patch_size=PATCH_SIZE):
    super().__init__()
    self.proj=nn.Conv2d(classes,embed_dim,kernel_size=patch_size,stride=patch_size)
  def forward(self,x):
    x=self.proj(x)
    x=x.flatten(2).transpose(1,2)


#ViT Class
